### Named Entity Recognition (spaCy)

In [3]:
import pandas as pd
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from transformers import pipeline
from sklearn.metrics import classification_report

In [4]:
ner_data = pd.read_csv("NER-test.tsv", sep="\t")
sentiment_topic_data = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")
print("Number of NER tokens:", len(ner_data))
display(ner_data.head())
print("Number of sentiment/topic sentences:", len(sentiment_topic_data))
display(sentiment_topic_data.head())

Number of NER tokens: 214


,sentence id,token id,token,BIO NER tag
0,0,0,It,O
1,0,1,took,O
2,0,2,eight,O
3,0,3,years,O
4,0,4,for,O



Number of sentiment/topic sentences: 10


,sentence id,text,sentiment,topic
0,0,It took eight years for Warner Brothers to rec...,negative,movie
1,1,All the New York University students love this...,positive,restaurant
2,2,This Italian place is really trendy but they h...,negative,restaurant
3,3,"In conclusion, my review of this book would be...",positive,book
4,4,The story of this movie is focused on Carl Bra...,neutral,movie


In [34]:

ner_data.columns = ner_data.columns.str.strip()
sentiment_topic_data.columns = sentiment_topic_data.columns.str.strip()

print("NER entity tags:")
print(ner_data[ner_data["BIO NER tag"] != "O"]["BIO NER tag"].value_counts())

print("\nSentiment distribution:")
print(sentiment_topic_data["sentiment"].value_counts())

print("\nTopic distribution:")
print(sentiment_topic_data["topic"].value_counts())




NER entity tags:
BIO NER tag
I-PER     8
B-PER     6
B-ORG     4
B-LOC     4
I-ORG     3
B-MISC    3
I-LOC     2
I-MISC    1
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    4
negative    3
neutral     3
Name: count, dtype: int64

Topic distribution:
topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64


In [5]:
sentences = {}

for _, row in ner_data.iterrows():

    sentence_id = row["sentence id"]
    token = row["token"]

    if sentence_id not in sentences:
        sentences[sentence_id] = token
    else:
        sentences[sentence_id] += " " + token

for sentence_id in sentences:
    print(sentence_id, ":", sentences[sentence_id])

0 : It took eight years for Warner Brothers to recover from the disaster that was this movie .
1 : All the New York University students love this diner in Soho so it makes for a fun young atmosphere .
2 : This Italian place is really trendy but they have forgotten about the most important part of a restaurant , the food .
3 : In conclusion , my review of this book would be : I like Jane Austen and understand why she is famous .
4 : The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African American deep sea diver in the Navy .
5 : Chris O'Donnell stated that while filming for this movie , he felt like he was in a toy commercial .
6 : My husband and I moved to Amsterdam 6 years ago and for as long as we have lived here , Blauwbrug has been our favorite place to eat !
7 : Dame Maggie Smith performed her role excellently , as she does in all her movies .
8 : The new movie by Mr. Kruno was shot in New York , but the story takes place in

In [6]:
nlp = spacy.load("en_core_web_sm")

In [7]:
results = []

for sentence_id in sentences:

    sentence = sentences[sentence_id]

    doc = nlp(sentence)

    for ent in doc.ents:

        results.append({"sentence_id": sentence_id, "entity": ent.text, "label": ent.label_})

ner_results = pd.DataFrame(results)

display(ner_results)

,sentence_id,entity,label
0,0,eight years,DATE
1,0,Warner Brothers,ORG
2,1,the New York University,ORG
3,1,Soho,LOC
4,2,Italian,NORP
5,3,Jane Austen,PERSON
6,4,Carl Brashear,PERSON
7,4,Cuba Gooding Jr.,PERSON
8,4,first,ORDINAL
9,4,African American,NORP


In [12]:
true_labels = []
predicted_labels = []

for sentence_id in sentences:

    sentence_rows = ner_data[ner_data["sentence id"] == sentence_id]

    words = list(sentence_rows["token"])
    true_tags = list(sentence_rows["BIO NER tag"])

    sentence = " ".join(words)
    doc = nlp(sentence)

    predicted_tags = ["O"] * len(words)

    for ent in doc.ents:

        entity_words = ent.text.split()
        entity_label = ent.label_

        for i in range(len(words)):

            if words[i:i + len(entity_words)] == entity_words:

                predicted_tags[i] = "B-" + entity_label

                for j in range(1, len(entity_words)):
                    predicted_tags[i + j] = "I-" + entity_label

    true_labels.extend(true_tags)
    predicted_labels.extend(predicted_tags)

In [19]:
print("True labels:", true_labels[:30])
print("Predicted labels:", predicted_labels[:30])

True labels: ['O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O']
Predicted labels: ['O', 'O', 'B-DATE', 'I-DATE', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O']


In [36]:
print(classification_report(true_labels, predicted_labels, zero_division=0))

              precision    recall  f1-score   support

      B-DATE       0.00      0.00      0.00         0
       B-GPE       0.00      0.00      0.00         0
       B-LOC       1.00      0.25      0.40         4
      B-MISC       0.00      0.00      0.00         3
      B-NORP       0.00      0.00      0.00         0
   B-ORDINAL       0.00      0.00      0.00         0
       B-ORG       0.67      0.50      0.57         4
       B-PER       0.00      0.00      0.00         6
    B-PERSON       0.00      0.00      0.00         0
      I-DATE       0.00      0.00      0.00         0
       I-GPE       0.00      0.00      0.00         0
       I-LOC       0.00      0.00      0.00         2
      I-MISC       0.00      0.00      0.00         1
      I-NORP       0.00      0.00      0.00         0
       I-ORG       0.75      1.00      0.86         3
       I-PER       0.00      0.00      0.00         8
    I-PERSON       0.00      0.00      0.00         0
           O       0.99    

The spaCy NER model achieved 85% token-level accuracy and a weighted F1-score of 0.86.   
However, the macro F1-score was much lower because many entity classes had only a few examples.

### Sentiment Analysis (VADER)

In [ ]:
analyzer = SentimentIntensityAnalyzer()

In [ ]:
results = []

for _, row in sentiment_topic_data.iterrows():

    sentence_id = row["sentence id"]
    text = row["text"]

    scores = analyzer.polarity_scores(text)

    compound = scores["compound"]

    if compound >= 0.05:
        predicted_sentiment = "positive"
    elif compound <= -0.05:
        predicted_sentiment = "negative"
    else:
        predicted_sentiment = "neutral"

    results.append({"sentence_id": sentence_id, "text": text, "true_sentiment": row["sentiment"], "predicted_sentiment": predicted_sentiment, "compound_score": compound})

sentiment_results = pd.DataFrame(results)

display(sentiment_results)

,sentence_id,text,true_sentiment,predicted_sentiment,compound_score
0,0,It took eight years for Warner Brothers to rec...,negative,negative,-0.6249
1,1,All the New York University students love this...,positive,positive,0.8176
2,2,This Italian place is really trendy but they h...,negative,positive,0.0745
3,3,"In conclusion, my review of this book would be...",positive,positive,0.3612
4,4,The story of this movie is focused on Carl Bra...,neutral,positive,0.6124
5,5,Chris O'Donnell stated that while filming for ...,neutral,positive,0.3612
6,6,My husband and I moved to Amsterdam 6 years ag...,positive,positive,0.5093
7,7,Dame Maggie Smith performed her role excellent...,positive,positive,0.6249
8,8,The new movie by Mr. Kruno was shot in New Yor...,neutral,neutral,0.0000
9,9,"I always have loved English novels, but I just...",negative,positive,0.3506


In [ ]:
correct = sentiment_results["true_sentiment"] == sentiment_results["predicted_sentiment"]

accuracy = correct.mean()

print("VADER Accuracy:", accuracy)

VADER Accuracy: 0.6


### Sentiment Analysis (TextBlob)

In [ ]:
textblob_results = []

for _, row in sentiment_topic_data.iterrows():

    sentence_id = row["sentence id"]
    text = row["text"]

    blob = TextBlob(text)

    polarity = blob.sentiment.polarity

    if polarity > 0:
        predicted_sentiment = "positive"
    elif polarity < 0:
        predicted_sentiment = "negative"
    else:
        predicted_sentiment = "neutral"

    textblob_results.append({"sentence_id": sentence_id, "text": text, "true_sentiment": row["sentiment"], "predicted_sentiment": predicted_sentiment, "polarity": polarity})

textblob_results = pd.DataFrame(textblob_results)

display(textblob_results)

,sentence_id,text,true_sentiment,predicted_sentiment,polarity
0,0,It took eight years for Warner Brothers to rec...,negative,neutral,0.000000
1,1,All the New York University students love this...,positive,positive,0.259091
2,2,This Italian place is really trendy but they h...,negative,positive,0.375000
3,3,"In conclusion, my review of this book would be...",positive,positive,0.500000
4,4,The story of this movie is focused on Carl Bra...,neutral,positive,0.090000
5,5,Chris O'Donnell stated that while filming for ...,neutral,neutral,0.000000
6,6,My husband and I moved to Amsterdam 6 years ag...,positive,positive,0.287500
7,7,Dame Maggie Smith performed her role excellent...,positive,positive,1.000000
8,8,The new movie by Mr. Kruno was shot in New Yor...,neutral,positive,0.136364
9,9,"I always have loved English novels, but I just...",negative,positive,0.350000


In [ ]:
correct = textblob_results["true_sentiment"] == textblob_results["predicted_sentiment"]

accuracy = correct.mean()

print("TextBlob Accuracy:", accuracy)

TextBlob Accuracy: 0.5


### Sentiment Analysis (DistilBERT)

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

Device set to use cpu


In [ ]:
distilbert_results = []

for _, row in sentiment_topic_data.iterrows():

    sentence_id = row["sentence id"]
    text = row["text"]

    prediction = classifier(text)[0]

    if prediction["label"] == "POSITIVE":
        predicted_sentiment = "positive"
    else:
        predicted_sentiment = "negative"

    distilbert_results.append({"sentence_id": sentence_id, "text": text, "true_sentiment": row["sentiment"], "predicted_sentiment": predicted_sentiment, "confidence": prediction["score"]})

distilbert_results = pd.DataFrame(distilbert_results)

display(distilbert_results)

,sentence_id,text,true_sentiment,predicted_sentiment,confidence
0,0,It took eight years for Warner Brothers to rec...,negative,negative,0.999268
1,1,All the New York University students love this...,positive,positive,0.998819
2,2,This Italian place is really trendy but they h...,negative,negative,0.998976
3,3,"In conclusion, my review of this book would be...",positive,positive,0.997400
4,4,The story of this movie is focused on Carl Bra...,neutral,positive,0.994012
5,5,Chris O'Donnell stated that while filming for ...,neutral,negative,0.999416
6,6,My husband and I moved to Amsterdam 6 years ag...,positive,positive,0.999431
7,7,Dame Maggie Smith performed her role excellent...,positive,positive,0.999840
8,8,The new movie by Mr. Kruno was shot in New Yor...,neutral,negative,0.927757
9,9,"I always have loved English novels, but I just...",negative,negative,0.996239


In [ ]:
correct = distilbert_results["true_sentiment"] == distilbert_results["predicted_sentiment"]

accuracy = correct.mean()

print("DistilBERT Accuracy:", accuracy)

DistilBERT Accuracy: 0.7


### Topic Classifier (Rule-based)

In [ ]:
topic_results = []

for _, row in sentiment_topic_data.iterrows():

    sentence_id = row["sentence id"]
    text = row["text"].lower()

    if "movie" in text or "film" in text:
        predicted_topic = "movie"

    elif "restaurant" in text or "diner" in text or "food" in text:
        predicted_topic = "restaurant"

    elif "book" in text or "novel" in text:
        predicted_topic = "book"

    else:
        predicted_topic = "unknown"

    topic_results.append({"sentence_id": sentence_id, "text": row["text"], "true_topic": row["topic"], "predicted_topic": predicted_topic})

topic_results = pd.DataFrame(topic_results)

display(topic_results)

,sentence_id,text,true_topic,predicted_topic
0,0,It took eight years for Warner Brothers to rec...,movie,movie
1,1,All the New York University students love this...,restaurant,restaurant
2,2,This Italian place is really trendy but they h...,restaurant,restaurant
3,3,"In conclusion, my review of this book would be...",book,book
4,4,The story of this movie is focused on Carl Bra...,movie,movie
5,5,Chris O'Donnell stated that while filming for ...,movie,movie
6,6,My husband and I moved to Amsterdam 6 years ag...,restaurant,unknown
7,7,Dame Maggie Smith performed her role excellent...,movie,movie
8,8,The new movie by Mr. Kruno was shot in New Yor...,movie,movie
9,9,"I always have loved English novels, but I just...",book,book


In [ ]:
correct = topic_results["true_topic"] == topic_results["predicted_topic"]

accuracy = correct.mean()

print("Topic Accuracy:", accuracy)

Topic Accuracy: 0.9


### Topic Classifier (gensim)

In [90]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora
import gensim

In [91]:
def prep(text):

    words = simple_preprocess(text)

    cleaned_words = []

    for word in words:
        if word not in STOPWORDS:
            cleaned_words.append(word)

    return cleaned_words

In [92]:
texts = []

for _, row in sentiment_topic_data.iterrows():

    text = row["text"]

    words = prep(text)

    texts.append(words)

dictionary = corpora.Dictionary(texts)

corpus = []

for words in texts:
    bow = dictionary.doc2bow(words)
    corpus.append(bow)

In [98]:
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=30,
    iterations=200
)

for topic_number in range(3):

    words = lda_model.show_topic(topic_number, topn=8)

    topic_words = []

    for word, score in words:
        topic_words.append(word)

    print("Topic", topic_number, ":", topic_words)

Topic 0 : ['movie', 'story', 'years', 'gooding', 'sea', 'jr', 'brashear', 'american']
Topic 1 : ['new', 'york', 'movie', 'like', 'angeles', 'kruno', 'takes', 'shot']
Topic 2 : ['place', 'years', 'husband', 'blauwbrug', 'eat', 'lived', 'amsterdam', 'moved']


In [99]:
gensim_results = []

for i in range(len(sentiment_topic_data)):

    sentence_id = sentiment_topic_data.loc[i, "sentence id"]
    text = sentiment_topic_data.loc[i, "text"]
    true_topic = sentiment_topic_data.loc[i, "topic"]

    words = prep(text)
    bow = dictionary.doc2bow(words)

    topic_scores = lda_model.get_document_topics(bow)

    best_topic = max(topic_scores, key=lambda x: x[1])[0]

    gensim_results.append({
        "sentence_id": sentence_id,
        "true_topic": true_topic,
        "gensim_topic_number": best_topic
    })

gensim_results = pd.DataFrame(gensim_results)

display(gensim_results)

,sentence_id,true_topic,gensim_topic_number
0,0,movie,0
1,1,restaurant,1
2,2,restaurant,2
3,3,book,2
4,4,movie,0
5,5,movie,1
6,6,restaurant,2
7,7,movie,2
8,8,movie,1
9,9,book,2


In [102]:
topic_map = {
    0: "movie",
    1: "restaurant",
    2: "restaurant"
}

gensim_results["predicted_topic"] = gensim_results["gensim_topic_number"].map(topic_map)

display(gensim_results)

,sentence_id,true_topic,gensim_topic_number,predicted_topic
0,0,movie,0,movie
1,1,restaurant,1,restaurant
2,2,restaurant,2,book
3,3,book,2,book
4,4,movie,0,movie
5,5,movie,1,restaurant
6,6,restaurant,2,book
7,7,movie,2,book
8,8,movie,1,restaurant
9,9,book,2,book


In [103]:
correct = gensim_results["true_topic"] == gensim_results["predicted_topic"]

accuracy = correct.mean()

print("Gensim LDA Accuracy:", accuracy)

Gensim LDA Accuracy: 0.5


### Topic Classifier (Zero-Shot Classification)

In [ ]:
zero_shot_results = []

labels = ["movie", "restaurant", "book"]

for _, row in sentiment_topic_data.iterrows():

    sentence_id = row["sentence id"]
    text = row["text"]

    result = classifier(text, candidate_labels=labels)

    predicted_topic = result["labels"][0]

    zero_shot_results.append({"sentence_id": sentence_id, "text": row["text"], "true_topic": row["topic"], "predicted_topic": predicted_topic})

zero_shot_results = pd.DataFrame(zero_shot_results)

display(zero_shot_results)

,sentence_id,text,true_topic,predicted_topic
0,0,It took eight years for Warner Brothers to rec...,movie,movie
1,1,All the New York University students love this...,restaurant,restaurant
2,2,This Italian place is really trendy but they h...,restaurant,restaurant
3,3,"In conclusion, my review of this book would be...",book,book
4,4,The story of this movie is focused on Carl Bra...,movie,movie
5,5,Chris O'Donnell stated that while filming for ...,movie,movie
6,6,My husband and I moved to Amsterdam 6 years ag...,restaurant,restaurant
7,7,Dame Maggie Smith performed her role excellent...,movie,movie
8,8,The new movie by Mr. Kruno was shot in New Yor...,movie,movie
9,9,"I always have loved English novels, but I just...",book,book


In [ ]:
correct = zero_shot_results["true_topic"] == zero_shot_results["predicted_topic"]

accuracy = correct.mean()

print("Zero-Shot Accuracy:", accuracy)

Zero-Shot Accuracy: 1.0


### Topic Classifier (TF-IDF + Logistic Regression)

In [104]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [109]:
texts = sentiment_topic_data["text"]
true_topics = sentiment_topic_data["topic"]

In [111]:
vectorizer = TfidfVectorizer()

text_features = vectorizer.fit_transform(texts)

In [112]:
model = LogisticRegression()

model.fit(text_features, true_topics)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [113]:
predicted_topics = model.predict(text_features)

In [114]:
tfidf_results = pd.DataFrame({
    "sentence_id": sentiment_topic_data["sentence id"],
    "true_topic": true_topics,
    "predicted_topic": predicted_topics
})

display(tfidf_results)

,sentence_id,true_topic,predicted_topic
0,0,movie,movie
1,1,restaurant,restaurant
2,2,restaurant,restaurant
3,3,book,movie
4,4,movie,movie
5,5,movie,movie
6,6,restaurant,restaurant
7,7,movie,movie
8,8,movie,movie
9,9,book,book


In [115]:
correct = tfidf_results["true_topic"] == tfidf_results["predicted_topic"]

accuracy = correct.mean()

print("TF-IDF Logistic Regression Accuracy:", accuracy)

TF-IDF Logistic Regression Accuracy: 0.9


### Project Description

The goal of this project was to apply Natural Language Processing (NLP) techniques to three text mining tasks: Named Entity Recognition (NER), Sentiment Analysis and Topic Classification.

Several approaches were evaluated, including rule-based systems, machine learning methods, and transformer-based language models. The objective was to compare their performance on the provided datasets and analyse their strengths and weaknesses.


### Dataset

Two datasets were provided for this project.

The NER dataset contained 214 tokens across 10 sentences. Each token was labelled using BIO NER tags that identify entities such as persons, organisations and locations.

The Sentiment and Topic dataset contained 10 sentences. Each sentence included a sentiment label (positive, negative or neutral) and a topic label (movie, restaurant or book).

The datasets were provided as TSV files and were used for testing and evaluation.


### Methodology

#### Named Entity Recognition

A pre-trained spaCy model (en_core_web_sm) was used to identify named entities. Predicted entities were converted into BIO tags and compared against the gold-standard annotations.

#### Sentiment Analysis

Three sentiment analysis systems were evaluated:

##### TextBlob
A lexicon-based system that predicts sentiment using polarity scores.

##### VADER
A rule-based sentiment analyser that calculates a compound sentiment score. Scores above 0.05 were classified as positive, scores below -0.05 as negative, and the remaining scores as neutral.

##### DistilBERT
A transformer-based language model that predicts sentiment using contextual information from the entire sentence.

#### Topic Classification

Three topic classification approaches were tested:

##### Rule-Based Classifier
Keywords such as "movie", "film", "restaurant", "food", "book" and "novel" were used to assign topics.

##### Gensim LDA
An unsupervised topic modelling approach that automatically discovers latent topics from the text.

##### TF-IDF + Logistic Regression
Text was converted into TF-IDF feature vectors and a Logistic Regression classifier was trained to predict topic labels.

### Results

| Task | System | Accuracy |
|--------|--------|--------|
| Named Entity Recognition | spaCy | 85% |
| Sentiment Analysis | TextBlob | 50% |
| Sentiment Analysis | VADER | 60% |
| Sentiment Analysis | DistilBERT | 70% |
| Topic Classification | Rule-Based | 90% |
| Topic Classification | Gensim LDA | 50% |
| Topic Classification | TF-IDF + Logistic Regression | 90% |

##### Main Findings

- spaCy achieved 85% token-level accuracy for NER.
- DistilBERT achieved the highest sentiment analysis accuracy (70%).
- VADER achieved moderate performance (60%).
- TextBlob achieved the lowest sentiment accuracy (50%).
- TF-IDF with Logistic Regression achieved the strongest supervised topic classification performance (90%).
- The Rule-Based classifier also achieved 90% accuracy.
- Gensim LDA performed less effectively due to the small dataset size.

### Comparison of Sentiment Systems

Three sentiment analysis approaches were evaluated.

##### TextBlob (50%)

TextBlob relies on word-level polarity scores. It often struggled with mixed opinions and complex sentence structures.

##### VADER (60%)

VADER improved performance through sentiment rules and a specialised sentiment lexicon. However, it still misclassified several sentences containing both positive and negative opinions.

##### DistilBERT (70%)

DistilBERT achieved the highest accuracy because it considers the context of the entire sentence rather than individual words.

Overall, DistilBERT was the most effective sentiment analysis model in this project.

# Error Analysis

## Named Entity Recognition

The spaCy model correctly identified many entities, including:

- Warner Brothers
- Jane Austen
- Carl Brashear
- New York
- Los Angeles

However, some errors were observed. For example, "Blauwbrug" was classified as a NORP entity instead of a location. The model also detected "Maggie Smith" but omitted the title "Dame".

## Sentiment Analysis

A challenging sentence was:

"I always have loved English novels, but I just couldn't get into this one."

The correct label was negative.

Both TextBlob and VADER predicted positive sentiment because they focused heavily on the word "loved". DistilBERT handled contextual information better but still made mistakes on some neutral examples.

## Topic Classification

The Rule-Based classifier depended heavily on predefined keywords. One restaurant sentence was classified as unknown because it did not contain expected keywords.

The TF-IDF + Logistic Regression classifier successfully learned word-topic relationships and achieved high accuracy.

The Gensim LDA model struggled because the dataset contained only 10 sentences, which is insufficient for reliable topic discovery.

### Error Analysis

#### Named Entity Recognition

The spaCy model correctly identified entities such as Jane Austen, Carl Brashear, Warner Brothers, New York University and Los Angeles.

Some mistakes were also observed. For example, "Blauwbrug" was classified as NORP instead of a location. The model also identified "Maggie Smith" but missed the title "Dame".  

The spaCy NER model was evaluated by comparing its predicted BIO tags with the manually annotated BIO labels. The model achieved a token-level accuracy of 85% and a weighted F1-score of 0.86. However, performance varied across entity types because several classes contained only a small number of examples in the dataset.

#### Sentiment Analysis

All three sentiment systems made mistakes on sentences that contained mixed opinions.

For example, the sentence:

"I always have loved English novels, but I just couldn't get into this one."

was labelled as negative in the dataset. However, TextBlob and VADER predicted positive sentiment because they focused on the word "loved" and did not fully understand the negative opinion later in the sentence.

DistilBERT handled sentence context better and achieved the highest accuracy. However, it still made mistakes on neutral sentences because the model was trained to predict only positive or negative sentiment.

#### Topic Classification

The rule based classifier achieved high accuracy but depended heavily on keywords.
One restaurant sentence was classified as "unknown" because it did not contain the words "restaurant", "food" or "diner" even though the sentence clearly referred to a place to eat.

The zero shot classifier achieved 100% accuracy on this dataset. Unlike the rule based classifier, it understood the meaning of the sentences and correctly classified the restaurant sentence that did not contain the words "restaurant", "food" or "diner".

TF-IDF converted the text into numerical feature vectors based on word importance. Logistic Regression was then trained using these features and the known topic labels. The model correctly classified 9 out of 10 sentences, achieving an accuracy of 90%. Only one book-related sentence was misclassified as a movie sentence. This performance was substantially better than the unsupervised Gensim LDA model because Logistic Regression directly learns from the labeled training examples.



### Conclusion

This project applied three NLP techniques to a small text mining dataset.

The spaCy NER model successfully identified most named entities including people, organisations and locations.

For sentiment analysis, DistilBERT achieved the best result with 70% accuracy. VADER achieved 60% accuracy and TextBlob achieved 50% accuracy.

The zero shot classifier achieved the highest overall performance with 100% accuracy. The rule based classifier also performed well with 90% accuracy.

These results show that more advanced NLP models can improve performance but simple rule based methods can still perform well on small datasets.
